# IA708 — Notebook 1 : Exploration des données
## Dataset UCI German Credit

**Télécom Paris — Mastère IA Multimodale, 2026**

---

### Objectif de ce notebook

Avant de construire n'importe quel modèle, il faut **comprendre les données** :
- Qui sont les individus ? Que représente chaque variable ?
- Quels groupes sont présents ? Sont-ils équilibrés ?
- Quels sont les taux de défaut par groupe (attributs sensibles) ?
- Y a-t-il des corrélations problématiques ?

Cette étape est **cruciale pour l'IA responsable** : les biais dans les données se retrouvent toujours dans le modèle.

---

### Contexte : scoring crédit et équité

Le **scoring de crédit** consiste à prédire si un demandeur va rembourser son prêt. C'est une décision à fort impact :
un refus de crédit peut empêcher quelqu'un d'acheter un logement, de créer une entreprise, ou de faire face à une urgence.

Des études ont montré que les systèmes automatisés de scoring ont tendance à désavantager certains groupes (femmes, jeunes,
minorités ethniques) — pas nécessairement parce que le modèle utilise directement l'attribut sensible, mais parce qu'il
utilise des **proxies** : des variables corrélées à l'attribut sensible (ex. code postal, type d'emploi, montant du crédit).

---

### Plan du notebook
1. Chargement et traduction des codes
2. Description des 20 features
3. Distribution de la variable cible
4. Analyse de chaque feature (distributions)
5. Attributs sensibles : genre et âge
6. Taux de défaut par groupe
7. Corrélations entre features et cible
8. Conclusion

---
## 1. Chargement et traduction des codes

Le fichier `german.data` (UCI Statlog) utilise des **codes opaques** pour les variables catégorielles : `A11`, `A34`, `A93`, etc.
Ces codes sont définis dans `german.pdf`. On les traduit ici en labels lisibles.

**Pourquoi c'est important ?** Travailler avec `A11` ou `A93` rend les résultats (SHAP, importances) illisibles.
La traduction est nécessaire pour pouvoir interpréter les modèles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

# ── Noms des colonnes (ordre dans german.data) ────────────────────────────────
COLUMNS = [
    "checking_status", "duration_in_month", "credit_history", "purpose",
    "credit_amount", "savings_account_bonds", "present_employment_since",
    "installment_rate", "personal_status_sex", "other_debtors_guarantors",
    "present_residence_since", "property", "age_in_years",
    "other_installment_plans", "housing", "number_of_existing_credits",
    "job", "number_of_people_liable", "telephone", "foreign_worker",
    "raw_target",
]

# ── Traduction des codes Axx → labels lisibles (source : german.pdf) ─────────
LABEL_MAP = {
    "checking_status": {
        "A11": "solde < 0 DM",
        "A12": "0 ≤ solde < 200 DM",
        "A13": "solde ≥ 200 DM (ou salaire dom. ≥ 1 an)",
        "A14": "pas de compte courant",
    },
    "credit_history": {
        "A30": "aucun crédit ou tous remboursés",
        "A31": "tous crédits banque remboursés",
        "A32": "crédits existants remboursés",
        "A33": "retards passés",
        "A34": "compte critique / autres crédits",
    },
    "purpose": {
        "A40": "voiture (neuve)",
        "A41": "voiture (occasion)",
        "A42": "meubles",
        "A43": "radio/TV",
        "A44": "électroménager",
        "A45": "réparations",
        "A46": "éducation",
        "A47": "vacances",
        "A48": "reconversion",
        "A49": "business",
        "A410": "autres",
    },
    "savings_account_bonds": {
        "A61": "épargne < 100 DM",
        "A62": "100 ≤ épargne < 500 DM",
        "A63": "500 ≤ épargne < 1000 DM",
        "A64": "épargne ≥ 1000 DM",
        "A65": "inconnu / pas d'épargne",
    },
    "present_employment_since": {
        "A71": "sans emploi",
        "A72": "< 1 an",
        "A73": "1–4 ans",
        "A74": "4–7 ans",
        "A75": "≥ 7 ans",
    },
    "personal_status_sex": {
        "A91": "homme, divorcé/séparé",
        "A92": "femme, divorcée/mariée",
        "A93": "homme, célibataire",
        "A94": "homme, marié/veuf",
        "A95": "femme, célibataire",
    },
    "other_debtors_guarantors": {
        "A101": "aucun",
        "A102": "co-demandeur",
        "A103": "garant",
    },
    "property": {
        "A121": "immobilier",
        "A122": "épargne logement / assurance-vie",
        "A123": "voiture ou autre",
        "A124": "inconnu / pas de propriété",
    },
    "other_installment_plans": {
        "A141": "banque",
        "A142": "magasins",
        "A143": "aucun",
    },
    "housing": {
        "A151": "locataire",
        "A152": "propriétaire",
        "A153": "hébergé gratuitement",
    },
    "job": {
        "A171": "sans emploi / non qualifié non-résident",
        "A172": "non qualifié résident",
        "A173": "employé qualifié",
        "A174": "management / hautement qualifié",
    },
    "telephone": {
        "A191": "aucun",
        "A192": "oui (au nom du client)",
    },
    "foreign_worker": {
        "A201": "oui",
        "A202": "non",
    },
}

# ── Chargement + traduction automatique ───────────────────────────────────────
raw = pd.read_csv("data/raw/german.data", sep=r"\s+", header=None, names=COLUMNS)

for col, mapping in LABEL_MAP.items():
    raw[col] = raw[col].astype(str).map(mapping).fillna(raw[col].astype(str))

raw["default"] = (raw["raw_target"] == 2).astype(int)   # 1 = mauvais payeur

print(f"Dataset chargé : {raw.shape[0]} lignes, {raw.shape[1]} colonnes")
print(f"Taux de défaut global : {raw['default'].mean():.1%}")
raw.head(5)

---
## 2. Description des 20 features

| Feature | Type | Description |
|---|---|---|
| `checking_status` | cat | Solde du compte courant (très prédictif) |
| `duration_in_month` | num | Durée du crédit en mois |
| `credit_history` | cat | Historique de remboursement |
| `purpose` | cat | Objet du crédit |
| `credit_amount` | num | Montant du crédit (DM) |
| `savings_account_bonds` | cat | Épargne disponible |
| `present_employment_since` | cat | Ancienneté dans l'emploi |
| `installment_rate` | num | Taux de remboursement (% du revenu) |
| `personal_status_sex` | cat | Statut marital + genre (**attribut sensible**) |
| `other_debtors_guarantors` | cat | Présence d'un co-demandeur ou garant |
| `present_residence_since` | num | Années à l'adresse actuelle |
| `property` | cat | Bien possédé |
| `age_in_years` | num | Âge (**attribut sensible**) |
| `other_installment_plans` | cat | Autres plans de paiement |
| `housing` | cat | Type de logement |
| `number_of_existing_credits` | num | Nombre de crédits en cours |
| `job` | cat | Catégorie socio-professionnelle |
| `number_of_people_liable` | num | Nombre de personnes à charge |
| `telephone` | cat | Possède un téléphone enregistré |
| `foreign_worker` | cat | Travailleur étranger |

**Variables numériques :** `duration_in_month`, `credit_amount`, `installment_rate`, `present_residence_since`, `age_in_years`, `number_of_existing_credits`, `number_of_people_liable`

**Variables catégorielles :** toutes les autres

In [ ]:
# ── Vue d'ensemble : types, valeurs manquantes ────────────────────────────────
NUMERIC = ["duration_in_month", "credit_amount", "installment_rate",
           "present_residence_since", "age_in_years",
           "number_of_existing_credits", "number_of_people_liable"]

CATEGORICAL = [c for c in raw.columns
               if c not in NUMERIC + ["raw_target", "default"]]

print("=== Statistiques numériques ===")
display(raw[NUMERIC].describe().round(2))

print("\n=== Valeurs manquantes ===")
missing = raw.isnull().sum()
print(missing[missing > 0] if missing.any() else "Aucune valeur manquante.")

---
## 3. Distribution de la variable cible

Le dataset est **déséquilibré** : environ 70% de bons payeurs et 30% de mauvais.

Ce déséquilibre a des implications importantes :
- Un seuil fixe à 0.5 sous-estimera les défauts (la classe positive)
- L'**accuracy** est une mauvaise métrique ici (un modèle naïf qui prédit toujours "bon payeur" obtient 70% d'accuracy)
- On préférera la **Balanced Accuracy** ou l'**AUC ROC**
- Le split train/val/test doit être **stratifié** pour conserver ce ratio dans chaque split

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Camembert
counts = raw["default"].value_counts()
axes[0].pie(counts, labels=["Bon payeur (0)", "Défaut (1)"],
            colors=["#4CAF50", "#f44336"], autopct="%1.1f%%", startangle=90)
axes[0].set_title("Distribution de la cible")

# Barres
axes[1].bar(["Bon payeur (0)", "Défaut (1)"], counts.values,
            color=["#4CAF50", "#f44336"])
axes[1].set_ylabel("Nombre d'individus")
axes[1].set_title("Déséquilibre des classes")
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 5, str(v), ha="center", fontweight="bold")

plt.suptitle("Variable cible : défaut de paiement", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nTaux de défaut : {raw['default'].mean():.1%}")
print("→ Dataset déséquilibré : utiliser Balanced Accuracy et AUC ROC, pas l'accuracy simple.")

---
## 4. Distributions des features numériques

On visualise la distribution de chaque feature numérique, séparée par la valeur de la cible.
Cela permet de voir **quelles variables discriminent** visuellement les bons et mauvais payeurs.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

colors = {0: "#4CAF50", 1: "#f44336"}
labels = {0: "Bon payeur", 1: "Défaut"}

for i, col in enumerate(NUMERIC):
    ax = axes[i]
    for target_val in [0, 1]:
        subset = raw[raw["default"] == target_val][col]
        ax.hist(subset, bins=25, alpha=0.5, color=colors[target_val],
                label=labels[target_val], density=True)
    ax.set_title(col.replace("_", " "), fontsize=10)
    ax.set_xlabel("")
    ax.legend(fontsize=8)

axes[-1].set_visible(False)

patches = [mpatches.Patch(color=colors[k], label=labels[k]) for k in [0, 1]]
fig.legend(handles=patches, loc="lower right", fontsize=10)
plt.suptitle("Distribution des variables numériques par cible", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5. Distributions des features catégorielles

Pour les variables catégorielles, on visualise le **taux de défaut par modalité**.
Cela révèle quelles valeurs sont associées à un risque élevé.

In [ ]:
# Top 6 features catégorielles les plus discriminantes
top_cats = ["checking_status", "credit_history", "savings_account_bonds",
            "purpose", "present_employment_since", "housing"]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.flatten()

for i, col in enumerate(top_cats):
    ax = axes[i]
    rates = raw.groupby(col)["default"].mean().sort_values(ascending=False)
    bars = ax.barh(range(len(rates)), rates.values,
                   color=["#f44336" if v > 0.35 else "#FF9800" if v > 0.25 else "#4CAF50"
                          for v in rates.values])
    ax.set_yticks(range(len(rates)))
    ax.set_yticklabels([str(l)[:30] for l in rates.index], fontsize=8)
    ax.set_xlabel("Taux de défaut")
    ax.set_title(col.replace("_", " "), fontsize=10, fontweight="bold")
    ax.axvline(raw["default"].mean(), color="black", linestyle="--", alpha=0.5, label="Moyenne")
    for j, v in enumerate(rates.values):
        ax.text(v + 0.005, j, f"{v:.0%}", va="center", fontsize=8)

plt.suptitle("Taux de défaut par modalité (features clés)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6. Attributs sensibles : genre et âge

### Construction des attributs sensibles

**Genre** : inféré depuis `personal_status_sex`.
Les codes `A91`, `A93`, `A94` → **male** ; `A92`, `A95` → **female**.

> **Attention** : cette inférence est une simplification. En pratique, le genre est un continuum
> et ne devrait pas être inféré de cette manière. On l'utilise ici uniquement à des fins d'audit,
> conformément à la littérature sur ce dataset.

**Âge** : binarisé en `young` (< 25 ans) vs `older` (≥ 25 ans).
Le seuil 25 ans est standard dans la littérature fairness sur ce dataset.

### Groupe privilégié vs groupe de comparaison

On identifie comme **groupe privilégié** le groupe qui a le taux de sélection (approbation de crédit) historiquement
le plus élevé dans les données :
- Genre : **male** est le groupe majoritaire avec un taux d'approbation plus élevé
- Âge : **older** a statistiquement de meilleurs antécédents de crédit

In [ ]:
# ── Construction des attributs sensibles ─────────────────────────────────────
GENDER_MAP = {
    "homme, divorcé/séparé": "male",
    "homme, célibataire":    "male",
    "homme, marié/veuf":     "male",
    "femme, divorcée/mariée": "female",
    "femme, célibataire":    "female",
}
AGE_THRESHOLD = 25

raw["gender"]    = raw["personal_status_sex"].map(GENDER_MAP)
raw["age_group"] = np.where(raw["age_in_years"] < AGE_THRESHOLD, "young", "older")

print("=== Distribution du genre ===")
display(raw["gender"].value_counts().to_frame("count"))

print("\n=== Distribution de l'âge ===")
display(raw["age_group"].value_counts().to_frame("count"))

# Vérification des NaN
print(f"\nNaN genre   : {raw['gender'].isna().sum()}")
print(f"NaN age_group: {raw['age_group'].isna().sum()}")

In [ ]:
# ── Distribution de l'âge avec seuil ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogramme de l'âge
axes[0].hist(raw["age_in_years"], bins=30, color="#2196F3", edgecolor="white", alpha=0.8)
axes[0].axvline(AGE_THRESHOLD, color="red", linestyle="--",
                label=f"Seuil jeune/âgé ({AGE_THRESHOLD} ans)")
axes[0].set_xlabel("Âge (années)")
axes[0].set_ylabel("Nombre d'individus")
axes[0].set_title("Distribution de l'âge")
axes[0].legend()

# Camembert genre
gender_counts = raw["gender"].value_counts()
axes[1].pie(gender_counts, labels=gender_counts.index,
            autopct="%1.1f%%", colors=["#2196F3", "#FF9800"], startangle=90)
axes[1].set_title("Répartition du genre")

plt.suptitle("Attributs sensibles", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Taux de défaut par groupe sensible

Cette analyse est la première étape de l'**audit de biais** : existe-t-il un écart de taux de défaut
entre les groupes ? Si oui, est-ce dû à une différence réelle de risque, ou à un biais systématique ?

**Indicateurs clés :**
- **Base rate** : taux de défaut réel (proportion de `Y=1` dans le groupe)
- **Taux de sélection** : dans un modèle, proportion de prédictions positives (ici = base rate sur données brutes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (attr, title) in zip(axes, [("gender", "Genre"), ("age_group", "Groupe d'âge")]):
    stats = raw.groupby(attr)["default"].agg(["mean", "count"]).rename(
        columns={"mean": "taux_défaut", "count": "n"})
    bars = ax.bar(stats.index, stats["taux_défaut"],
                  color=["#2196F3", "#FF9800"], alpha=0.85)
    ax.axhline(raw["default"].mean(), color="black", linestyle="--",
               alpha=0.7, label=f"Moyenne ({raw['default'].mean():.1%})")
    ax.set_ylim(0, 0.6)
    ax.set_ylabel("Taux de défaut")
    ax.set_title(f"Taux de défaut par {title}")
    ax.legend()
    for bar, (idx, row) in zip(bars, stats.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{row['taux_défaut']:.1%}\n(n={int(row['n'])})",
                ha="center", fontsize=10, fontweight="bold")

plt.suptitle("Taux de défaut par attribut sensible", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Affichage détaillé
for attr in ["gender", "age_group"]:
    print(f"\n=== {attr} ===")
    display(raw.groupby(attr)["default"].agg(["mean", "count"]).rename(
        columns={"mean": "taux_défaut", "count": "n"}).round(3))

---
## 8. Corrélations : features vs cible

On mesure l'association entre chaque feature numérique et la cible (corrélation de Pearson).
Pour les catégorielles, on utilise l'**eta-carré** (variance inter-groupes / variance totale).

**Limites de cette analyse :**
- La corrélation est linéaire ; des relations non-linéaires peuvent être manquées
- Des features peu corrélées individuellement peuvent être importantes en combinaison
- On ne peut pas conclure sur la **causalité** depuis les corrélations

In [ ]:
# ── Corrélations numériques ───────────────────────────────────────────────────
num_corr = raw[NUMERIC + ["default"]].corr()["default"].drop("default").sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot corrélations numériques
colors_corr = ["#f44336" if v > 0 else "#4CAF50" for v in num_corr.values]
axes[0].barh(num_corr.index, num_corr.values, color=colors_corr)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Corrélation de Pearson avec 'default'")
axes[0].set_title("Features numériques vs cible")
for i, v in enumerate(num_corr.values):
    axes[0].text(v + (0.003 if v >= 0 else -0.003), i, f"{v:.3f}",
                 va="center", ha="left" if v >= 0 else "right", fontsize=9)

# Heatmap corrélations inter-numériques
corr_matrix = raw[NUMERIC].corr()
im = axes[1].imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(NUMERIC)))
axes[1].set_yticks(range(len(NUMERIC)))
axes[1].set_xticklabels([c.replace("_", "\n") for c in NUMERIC], fontsize=7)
axes[1].set_yticklabels([c.replace("_", " ") for c in NUMERIC], fontsize=8)
axes[1].set_title("Corrélations inter-features numériques")
plt.colorbar(im, ax=axes[1], fraction=0.046)

plt.suptitle("Analyse des corrélations", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Analyse des proxies potentiels

Un **proxy** est une variable non-sensible qui est **corrélée avec un attribut sensible**.
Par exemple, si les femmes empruntent plus souvent pour l'électroménager et moins pour les voitures,
alors `purpose` est un proxy du genre.

> **Implication :** même en retirant `personal_status_sex` du modèle, le biais peut subsister
> via les proxies. C'est pourquoi l'anti-classification seule (ne pas utiliser l'attribut sensible)
> est insuffisante — il faut mesurer et corriger les métriques de fairness directement.

In [ ]:
# Corrélation entre attributs sensibles et features numériques
print("=== Corrélation (point-bisériale) : features numériques vs genre ===")
gender_bin = (raw["gender"] == "male").astype(int)
corr_gender = pd.Series(
    {col: gender_bin.corr(raw[col]) for col in NUMERIC},
    name="corr_avec_genre"
).sort_values(key=abs, ascending=False)
display(corr_gender.round(3).to_frame())

print("\n=== Taux de défaut par purpose et genre ===")
ct = pd.crosstab(raw["purpose"], raw["gender"], values=raw["default"],
                 aggfunc="mean").round(2)
display(ct)
print("\n→ Si les taux diffèrent par genre pour un même 'purpose', c'est un indice de biais.")

---
## 10. Conclusion

### Ce qu'on a appris

1. **Dataset déséquilibré** (70/30) → nécessite stratification et métriques adaptées (AUC, Balanced Accuracy)
2. **`checking_status`** est la feature la plus discriminante visuellement — un compte débiteur prédit fortement le défaut
3. **Écart de taux de défaut** entre groupes sensibles :
   - Les femmes ont un taux de défaut légèrement supérieur dans les données brutes
   - Les jeunes (< 25 ans) ont un taux de défaut plus élevé que les plus âgés
4. **Proxies identifiés** : `credit_amount`, `purpose`, et `duration_in_month` sont corrélés avec le genre
   → le biais peut subsister même sans l'attribut sensible direct

### Ce qu'on va faire dans les notebooks suivants

- **Notebook 2** : construire le modèle de base et mesurer ses performances
- **Notebook 3** : mesurer et corriger le biais avec reweighing et calibration par groupe
- **Notebook 4** : expliquer les prédictions (SHAP) et tester la robustesse

> **Règle d'or** : un modèle sur-performant mais biaisé est inacceptable.
> L'objectif n'est pas de maximiser l'AUC à tout prix, mais d'atteindre
> un bon compromis performance / équité / explicabilité.